# Clinical Document Generator / Demo do Fluxo Principal

Este notebook demonstra o fluxo completo da API de prontuários eletrônicos:

1. **Criação de EHR** - cadastro de um prontuário com dados do paciente
2. **Transcrição de áudio** - envio de arquivo de áudio para transcrição via Gemini
3. **Geração de documento clínico** - geração de nota SOAP, prescrição ou encaminhamento via Gemini (Google GenAI)
4. **Edição de documento** - revisão manual do conteúdo gerado

> **Pré-requisito:** o servidor deve estar rodando em `http://localhost:8000`.
 
> Execute `python manage.py runserver` no diretório `backend/.`

In [ ]:
import requests
import json

BASE_URL = "http://localhost:8000/api"

def pp(data):
    """Pretty-print JSON response."""
    print(json.dumps(data, indent=2, ensure_ascii=False))

## 1. Criação de EHR

Cria um novo prontuário eletrônico com dados do paciente e transcrição da consulta.

In [ ]:
ehr_payload = {
    "patient_name": "Maria da Silva",
    "consultation_type": "presencial",
    "transcription": (
        "Paciente do sexo feminino, 45 anos, comparece à consulta relatando dor de cabeça "
        "intensa há 3 dias, com piora ao acordar. Nega febre. Refere uso esporádico de "
        "paracetamol sem melhora significativa. Exame físico: PA 130/85mmHg, FC 78bpm. "
        "Sem sinais de irritação meníngea. Hipótese diagnóstica: cefaleia tensional."
    ),
    "extra": {"specialty": "clínica geral", "insurance": "SUS"},
}

resp = requests.post(f"{BASE_URL}/ehrs/", json=ehr_payload)
resp.raise_for_status()

ehr = resp.json()
ehr_id = ehr["id"]
print(f"EHR criado com ID: {ehr_id}")
pp(ehr)

### Verificando na listagem

In [ ]:
resp = requests.get(f"{BASE_URL}/ehrs/")
resp.raise_for_status()
data = resp.json()
print(f"Total de EHRs: {data['count']}")
pp(data["results"][:3])

## 2. Transcrição de Áudio

Envia um arquivo de áudio para transcrição automática via Gemini.

> **Nota:** substitua o caminho abaixo por um arquivo `.mp3` ou `.wav` real para testar
> com a API do Gemini. Este exemplo usa um placeholder.

In [ ]:
# Descomente e ajuste o caminho para testar com áudio real:
#
# audio_path = "../samples/consulta.mp3"
# with open(audio_path, "rb") as f:
#     resp = requests.post(
#         f"{BASE_URL}/transcriptions/",
#         files={"audio": ("consulta.mp3", f, "audio/mpeg")},
#     )
#     resp.raise_for_status()
#     transcription = resp.json()
#     print(f"Transcrição ({transcription['provider']}/{transcription['model']}):")
#     print(transcription["text"])

print("(Transcrição pulada - descomente o bloco acima com um arquivo de áudio real)")

## 3. Geração de Documento Clínico

Usa a transcrição do EHR para gerar um documento clínico via Gemini (Google GenAI).

Templates disponíveis: `soap_note`, `prescription`, `referral`.

In [ ]:
gen_payload = {"template_identifier": "soap_note"}

resp = requests.post(f"{BASE_URL}/ehrs/{ehr_id}/generate/", json=gen_payload)
resp.raise_for_status()

doc = resp.json()
doc_id = doc["id"]
print(f"Documento gerado - ID: {doc_id}, template: {doc['template_identifier']}")
print("\n--- Conteúdo gerado ---")
print(doc["content"])

### 3b. Geração via Streaming (SSE) — Opcional

O endpoint `/generate/stream/` retorna Server-Sent Events com chunks de texto em tempo real.

In [ ]:
gen_payload = {"template_identifier": "prescription"}

resp = requests.post(
    f"{BASE_URL}/ehrs/{ehr_id}/generate/stream/",
    json=gen_payload,
    stream=True,
    headers={"Accept": "text/event-stream"},
)
resp.raise_for_status()

full_content = ""
stream_doc_id = None

for line in resp.iter_lines(decode_unicode=True):
    if not line or not line.startswith("data: "):
        continue
    payload = json.loads(line[len("data: "):])

    if payload["type"] == "chunk":
        print(payload["content"], end="", flush=True)
        full_content += payload["content"]
    elif payload["type"] == "done":
        stream_doc_id = payload["document_id"]
        print(f"\n\n✔ Documento salvo: {stream_doc_id}")
    elif payload["type"] == "error":
        print(f"\n✘ Erro: {payload['detail']}")

## 4. Edição de Documento

O médico revisa e edita o conteúdo gerado pela IA.

In [ ]:
edit_payload = {
    "content": (
        "S: Paciente feminina, 45a, cefaleia intensa há 3 dias, piora matinal. "
        "Nega febre. Paracetamol sem melhora.\n"
        "O: PA 130/85, FC 78. Sem sinais meníngeos.\n"
        "A: Cefaleia tensional\n"
        "P: Ibuprofeno 400mg 8/8h por 5 dias. Retorno em 7 dias se persistir."
    ),
}

resp = requests.patch(
    f"{BASE_URL}/ehrs/{ehr_id}/documents/{doc_id}/",
    json=edit_payload,
)
resp.raise_for_status()

updated = resp.json()
print("Documento atualizado:")
pp(updated)

### Consultando o detalhe do EHR com documentos

In [ ]:
resp = requests.get(f"{BASE_URL}/ehrs/{ehr_id}/")
resp.raise_for_status()
detail = resp.json()

print(f"EHR: {detail['patient_name']} - {len(detail['documents'])} documento(s)")
for d in detail["documents"]:
    print(f"  [{d['template_identifier']}] {d['id'][:8]}…")

---

**Fim da demonstração.** Consulte o README do projeto para instruções completas de setup e execução.